# Runbook 4.3.b: Antivirus and endpoint protection

**Skill:** Mitigate Risk · **Maps to:** 4.3.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Endpoint Protection Platforms (EPP) use a mixture of signature detection (known hashes) and behavioral heuristics (actions taken by the process, such as excessive file updates or network socket connections) to identify and suspend threats.

You are tasked with writing a process-auditing rule engine `solve(data)`. The engine evaluates a list of running processes against:
1. A database of known malicious file hashes (signatures).
2. Max thresholds on process behavior: `max_file_modifications` and `max_network_connections`.

The function must return a list of PIDs of all processes that violate any of these rules.

**Lesson objective.** Explain the difference between signature-based and heuristic/behavioral malware detection, describe the function of host-based firewalls, and explain how endpoint protection platforms (EPP) safeguard devices in an enterprise.

### What comes in
A dictionary containing security signatures, behavioral thresholds, and a list of active processes:
```json
{
  "signatures": ["d41d8cd98f00b204e9800998ecf8427e", "098f6bcd4621d373cade4e832627b4f6"],
  "behavioral_rules": {
    "max_file_modifications": 10,
    "max_network_connections": 5
  },
  "processes": [
    {
      "pid": 101,
      "name": "backup_agent.exe",
      "hash": "81dc9bdb52d04dc20036dbd8313ed055",
      "file_modifications": 8,
      "network_connections": 2
    },
    {
      "pid": 102,
      "name": "ransomware.exe",
      "hash": "d41d8cd98f00b204e9800998ecf8427e",
      "file_modifications": 2,
      "network_connections": 0
    },
    {
      "pid": 103,
      "name": "spam_sender.exe",
      "hash": "1f10c9aa0090ff91e90ff8a8bc8a9bcf",
      "file_modifications": 1,
      "network_connections": 12
    }
  ]
}
```

### What must come out
A list of integers representing the PIDs of flagged processes, sorted in ascending order.
For the example input above:
```json
[102, 103]
```
*(Explanation: PID 102 matches a known signature; PID 103 exceeds the network connections threshold)*

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.


### Rules to respect
- [ ] Signature matching is case-insensitive (e.g., `"A1B2"` matches `"a1b2"`).
- [ ] Behavioral violations occur when a metric is *strictly greater* than the threshold (i.e. `file_modifications > max_file_modifications`).
- [ ] Standard library only.
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(data: dict) -> list[int]
```

#### Input (`data`)
A dictionary containing:
- `"signatures"`: list of strings (MD5 hashes of known malware)
- `"behavioral_rules"`: dict with keys:
  - `"max_file_modifications"` (int)
  - `"max_network_connections"` (int)
- `"processes"`: list of dicts, where each dict has:
  - `"pid"` (int)
  - `"name"` (string)
  - `"hash"` (string)
  - `"file_modifications"` (int)
  - `"network_connections"` (int)

#### Output (`result`)
A list of integers representing the PIDs of flagged processes.
A process is flagged if:
1. Its hash is in the `"signatures"` list.
2. Its `"file_modifications"` is strictly greater than `"max_file_modifications"`.
3. Its `"network_connections"` is strictly greater than `"max_network_connections"`.

The output list of PIDs must be sorted in ascending order. If no processes are flagged, returns `[]`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only.
- Case-insensitivity: MD5 hash comparisons should be case-insensitive (e.g., `"A1B2"` matches `"a1b2"`).

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 4.3.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""


def solve(data: dict) -> list:
    """Flag processes that match a known-malware signature or exceed behavioral thresholds.

    Args:
        data: dict with keys "signatures" (list[str]), "behavioral_rules" (dict with
            "max_file_modifications" and "max_network_connections"), and "processes"
            (list of dicts with pid, name, hash, file_modifications, network_connections).

    Returns:
        Sorted list of PIDs (int) for every process that matches a signature or
        exceeds a behavioral threshold.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_example_1_prompt_scenario():
    data = {
        "signatures": ["d41d8cd98f00b204e9800998ecf8427e", "098f6bcd4621d373cade4e832627b4f6"],
        "behavioral_rules": {"max_file_modifications": 10, "max_network_connections": 5},
        "processes": [
            {"pid": 101, "name": "backup_agent.exe", "hash": "81dc9bdb52d04dc20036dbd8313ed055",
             "file_modifications": 8, "network_connections": 2},
            {"pid": 102, "name": "ransomware.exe", "hash": "d41d8cd98f00b204e9800998ecf8427e",
             "file_modifications": 2, "network_connections": 0},
            {"pid": 103, "name": "spam_sender.exe", "hash": "1f10c9aa0090ff91e90ff8a8bc8a9bcf",
             "file_modifications": 1, "network_connections": 12},
        ],
    }
    assert impl.solve(data) == [102, 103]


def test_example_2_no_flags():
    data = {
        "signatures": ["aaaa"],
        "behavioral_rules": {"max_file_modifications": 10, "max_network_connections": 5},
        "processes": [
            {"pid": 1, "name": "clean.exe", "hash": "bbbb",
             "file_modifications": 3, "network_connections": 1},
        ],
    }
    assert impl.solve(data) == []


def test_edge_empty_process_list():
    data = {
        "signatures": ["aaaa"],
        "behavioral_rules": {"max_file_modifications": 10, "max_network_connections": 5},
        "processes": [],
    }
    assert impl.solve(data) == []


def test_signature_case_insensitive():
    data = {
        "signatures": ["A1B2C3"],
        "behavioral_rules": {"max_file_modifications": 100, "max_network_connections": 100},
        "processes": [
            {"pid": 5, "name": "evil.exe", "hash": "a1b2c3",
             "file_modifications": 0, "network_connections": 0},
        ],
    }
    assert impl.solve(data) == [5]


def test_boundary_thresholds_not_exceeded_when_equal():
    data = {
        "signatures": [],
        "behavioral_rules": {"max_file_modifications": 10, "max_network_connections": 5},
        "processes": [
            {"pid": 7, "name": "borderline.exe", "hash": "zzzz",
             "file_modifications": 10, "network_connections": 5},
        ],
    }
    assert impl.solve(data) == []


def test_boundary_thresholds_exceeded_by_one():
    data = {
        "signatures": [],
        "behavioral_rules": {"max_file_modifications": 10, "max_network_connections": 5},
        "processes": [
            {"pid": 8, "name": "over.exe", "hash": "zzzz",
             "file_modifications": 11, "network_connections": 0},
            {"pid": 9, "name": "over2.exe", "hash": "zzzz",
             "file_modifications": 0, "network_connections": 6},
        ],
    }
    assert impl.solve(data) == [8, 9]


def test_result_sorted_ascending_regardless_of_input_order():
    data = {
        "signatures": ["hash1", "hash2", "hash3"],
        "behavioral_rules": {"max_file_modifications": 999, "max_network_connections": 999},
        "processes": [
            {"pid": 30, "name": "c.exe", "hash": "hash3",
             "file_modifications": 0, "network_connections": 0},
            {"pid": 10, "name": "a.exe", "hash": "hash1",
             "file_modifications": 0, "network_connections": 0},
            {"pid": 20, "name": "b.exe", "hash": "hash2",
             "file_modifications": 0, "network_connections": 0},
        ],
    }
    assert impl.solve(data) == [10, 20, 30]


def test_multiple_reasons_same_pid_counted_once():
    data = {
        "signatures": ["deadbeef"],
        "behavioral_rules": {"max_file_modifications": 1, "max_network_connections": 1},
        "processes": [
            {"pid": 42, "name": "multi.exe", "hash": "deadbeef",
             "file_modifications": 50, "network_connections": 50},
        ],
    }
    assert impl.solve(data) == [42]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it